In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
import random

dbutils.widgets.text(
    "namespace",
    "evhua5816bd"
)

dbutils.widgets.text(
    "bronze_checkpoint",
    "abfss://tristar-team@dlsua5816bd.dfs.core.windows.net/raw/streaming/streaming-metadata/eventhub-checkpoint"
)

dbutils.widgets.text(
    "bronze_table",
    "dbr_dev_ua5816bd.team_tristar_bronze.vehicles_streaming"
)

NAMESPACE = dbutils.widgets.get("namespace")

EVENT_HUB_NAME = dbutils.secrets.get(
    scope="artem-gulidov-scope",
    key="artem-gulidov-eventhub-name"
)

EH_CONN_STR = dbutils.secrets.get(
    scope="artem-gulidov-scope",
    key="artem-gulidov-eventhub-connstr-listen"
)

kafka_options = {
    "kafka.bootstrap.servers":
        f"{NAMESPACE}.servicebus.windows.net:9093",
    "subscribe":
        EVENT_HUB_NAME,
    "kafka.security.protocol":
        "SASL_SSL",
    "kafka.sasl.mechanism":
        "PLAIN",
    "kafka.sasl.jaas.config":
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule '
        f'required username="$ConnectionString" '
        f'password="{EH_CONN_STR}";',
    "startingOffsets": "latest",
    "failOnDataLoss": "false"
}

In [0]:
events = (
    spark.readStream
        .format("kafka")
        .options(**kafka_options)
        .option("startingOffsets", "latest")
        .load()
)

json_events = events.select(
    F.col("value").cast("string").alias("json")
)

schema = StructType([
    StructField("generated", StringType()),
    StructField("routeShortName", StringType()),
    StructField("tripId", LongType()),
    StructField("routeId", LongType()),
    StructField("headsign", StringType()),
    StructField("vehicleCode", StringType()),
    StructField("vehicleService", StringType()),
    StructField("vehicleId", LongType()),
    StructField("speed", IntegerType()),
    StructField("direction", IntegerType()),
    StructField("delay", IntegerType()),
    StructField("scheduledTripStartTime", StringType()),
    StructField("lat", DoubleType()),
    StructField("lon", DoubleType()),
    StructField("gpsQuality", IntegerType())
])

parsed = (
    json_events.select(
        F.from_json(
            F.col("json"),
            schema
        ).alias("data")
    )
)

vehicles = parsed.select(
    F.col("data.generated").alias("generated"),
    F.col("data.routeShortName").alias("route_short_name"),
    F.col("data.tripId").alias("trip_id"),
    F.col("data.routeId").alias("route_id"),
    F.col("data.headsign").alias("headsign"),
    F.col("data.vehicleCode").alias("vehicle_code"),
    F.col("data.vehicleService").alias("vehicle_service"),
    F.col("data.vehicleId").alias("vehicle_id"),
    F.col("data.speed").alias("speed"),
    F.col("data.direction").alias("direction"),
    F.col("data.delay").alias("delay"),
    F.col("data.scheduledTripStartTime").alias(
        "scheduled_trip_start_time"
    ),
    F.col("data.lat").alias("latitude"),
    F.col("data.lon").alias("longitude"),
    F.col("data.gpsQuality").alias("gps_quality")
)

bronze_table = dbutils.widgets.get("bronze_table")
bronze_checkpoint = dbutils.widgets.get("bronze_checkpoint")

schema_evolution_columns = [
    "surprise_column_1",
    "surprise_column_2",
    "surprise_column_3"
]

def process_batch(batch_df, batch_id):
    if random.random() < 0.1:
        new_col = random.choice(schema_evolution_columns)

        batch_df = batch_df.withColumn(
            new_col,
            F.lit(random.randint(1, 100))
        )

        print(f"Added column {new_col}")

    (
        batch_df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(bronze_table)
    )

query = (
    vehicles.writeStream
        .foreachBatch(process_batch)
        .option(
            "checkpointLocation", bronze_checkpoint)
        .trigger(processingTime="5 seconds")
        .start()
)